# Week 9: Report Generator

SiteLens AI — Audience-specific translation of structured damage records via Gemini 2.5 Flash.

**Topic:** Prompt engineering — system instruction / user content split, hallucination floor, deterministic pre-computation  
**Dates:** May 17–21 2026  
**Status:** Complete

| # | Section | Description | Status |
|---|---------|-------------|--------|
| 1 | Setup | Env check, imports, data load | ✓ |
| 2 | Record selection | Three cases: fire-zone destroyed, seismic-only destroyed, survived | ✓ |
| 3 | Insurance (J-PIC) | Per-building damage assessment — full hallucination check | ✓ |
| 4 | Insurance sanity grid | Five-criterion pass/fail table across three records | ✓ |
| 5 | Engineering | Structural observation, failure mode, next-step | ✓ |
| 6 | Engineering sanity grid | Four-criterion check across three records | ✓ |
| 7 | Legal | Defensible factual record with explicit limitations | ✓ |
| 8 | Legal sanity grid | Statement-class and limitations check | ✓ |

In [ ]:
import os
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY not set"
print("API key present.")

In [ ]:
import pandas as pd
from src.translation.audience_translator import translate

df = pd.read_csv("../data/noto_crops/labels.csv")
print(f"Loaded {len(df):,} records. Columns: {list(df.columns)}")

In [ ]:
# Pick three records spanning the interesting cases
test_records = []

# (a) destroyed in fire zone — Asaichi market area
test_records.append(
    df[(df.damage_val == 1) & (df.gsi_fire == 1)].iloc[0].to_dict()
)
# (b) destroyed outside fire zone — seismic-only
test_records.append(
    df[(df.damage_val == 1) & (df.gsi_fire == 0)].iloc[0].to_dict()
)
# (c) survived
test_records.append(
    df[df.damage_val == 0].iloc[0].to_dict()
)

for i, rec in enumerate(test_records, 1):
    print(f"[{i}] s_fid={rec['s_fid']}  damage_val={rec['damage_val']}  "
          f"fire={rec['gsi_fire']}  tsunami={rec['gsi_tsunami']}  "
          f"slope={rec['gsi_slope_failure']}  conf={rec['conf']}")

## 3 — Insurance audience (J-PIC)

Pass criteria: J-PIC category matches damage_val; peril attribution from pre-computed flags; evidence basis matches conf; no invented facts; next-action from pre-computed damage_val.

In [ ]:
insurance_results = [translate(rec, audience="insurance") for rec in test_records]

for rec, result in zip(test_records, insurance_results):
    print("\n" + "=" * 72)
    print(f"s_fid {rec['s_fid']} — damage_val {rec['damage_val']} "
          f"— fire {rec['gsi_fire']} — seismic-only {1 - rec['gsi_fire']}")
    print("=" * 72)
    print("\n--- INSURANCE ADJUSTER ---")
    print(result["output_text"])

## 4 — Insurance sanity grid

Five criteria across three records. All cells should be consistent with the input flags and damage_val.

In [ ]:
import re

def extract(text, label):
    m = re.search(rf"{re.escape(label)}:\s*([^\n]+)", text)
    return m.group(1).strip() if m else "—"

ins_checks = []
for rec, result in zip(test_records, insurance_results):
    txt = result["output_text"]
    ins_checks.append({
        "s_fid":       rec["s_fid"][-6:],
        "damage_val":  rec["damage_val"],
        "gsi_fire":    rec["gsi_fire"],
        "J-PIC":       extract(txt, "Damage classification"),
        "primary":     extract(txt, "Primary peril"),
        "secondary":   extract(txt, "Secondary peril"),
        "evidence":    extract(txt, "Evidence basis"),
        "next_action": extract(txt, "Recommended next-action"),
    })

pd.DataFrame(ins_checks)

## 5 — Engineering audience

Pass criteria: observed condition matches damage_val; failure mode derived from hazard flags only; no structural details invented beyond the input; next-step appropriate to condition.

In [ ]:
engineering_results = [translate(rec, audience="engineering") for rec in test_records]

for rec, result in zip(test_records, engineering_results):
    print("\n" + "=" * 72)
    print(f"s_fid {rec['s_fid']} — damage_val {rec['damage_val']} "
          f"— fire {rec['gsi_fire']} — seismic-only {1 - rec['gsi_fire']}")
    print("=" * 72)
    print(result["output_text"])

## 6 — Engineering sanity grid

Four criteria: observed condition, failure mode source, evidence basis, recommended next-step.

In [ ]:
eng_checks = []
for rec, result in zip(test_records, engineering_results):
    txt = result["output_text"]
    eng_checks.append({
        "s_fid":        rec["s_fid"][-6:],
        "damage_val":   rec["damage_val"],
        "gsi_fire":     rec["gsi_fire"],
        "condition":    extract(txt, "Observed condition"),
        "failure_mode": extract(txt, "Likely failure mode"),
        "evidence":     extract(txt, "Evidence basis"),
        "next_step":    extract(txt, "Recommended next-step"),
    })

pd.DataFrame(eng_checks)

## 7 — Legal audience

Pass criteria: every assertion references a named source; statement classes enumerated; limitations listed explicitly; no probabilities or causal speculation.

In [ ]:
legal_results = [translate(rec, audience="legal") for rec in test_records]

for rec, result in zip(test_records, legal_results):
    print("\n" + "=" * 72)
    print(f"s_fid {rec['s_fid']} — damage_val {rec['damage_val']} "
          f"— fire {rec['gsi_fire']} — seismic-only {1 - rec['gsi_fire']}")
    print("=" * 72)
    print(result["output_text"])

## 8 — Legal sanity grid

Checks statement classes, limitations count, and attribution source across three records.

In [ ]:
legal_checks = []
for rec, result in zip(test_records, legal_results):
    txt = result["output_text"]
    n_limitations = len(re.findall(r"^\s*\d+[\.\)]", txt, re.MULTILINE))
    legal_checks.append({
        "s_fid":           rec["s_fid"][-6:],
        "damage_val":      rec["damage_val"],
        "gsi_fire":        rec["gsi_fire"],
        "condition":       extract(txt, "Documented condition"),
        "attribution_src": extract(txt, "Source of attribution"),
        "stmt_classes":    extract(txt, "Statement classes used"),
        "n_limitations":   n_limitations,
    })

pd.DataFrame(legal_checks)

## 9 — Pass/fail harnesses

Automated regression checks for all three audiences. Legal at temperature=0.0 is the strictest regression: identical input should produce identical output across runs — any change flags model-version drift.

In [ ]:
# --- Insurance harness ---

def expected_jpic(rec):
    return "全損" if rec["damage_val"] == 1 else "損害なし"

def expected_primary(rec):
    if rec["damage_val"] == 0:
        return "not applicable"
    if rec["gsi_fire"]:
        return "fire"
    if rec["gsi_tsunami"]:
        return "tsunami"
    if rec["gsi_slope_failure"]:
        return "slope_failure"
    return "seismic"

def expected_next_action(rec):
    return "field inspection" if rec["damage_val"] == 1 else "desk approval"

ins_grades = []
for rec, result in zip(test_records, insurance_results):
    txt = result["output_text"]
    row = {
        "s_fid":            rec["s_fid"][-6:],
        "jpic_pass":        expected_jpic(rec) in extract(txt, "Damage classification"),
        "primary_pass":     expected_primary(rec) in extract(txt, "Primary peril").lower(),
        "evidence_pass":    rec["conf"] in extract(txt, "Evidence basis").lower(),
        "next_action_pass": expected_next_action(rec) in extract(txt, "Recommended next-action").lower(),
    }
    row["all_pass"] = all(v for k, v in row.items() if k.endswith("_pass"))
    ins_grades.append(row)

ins_grades_df = pd.DataFrame(ins_grades)
print(f"Insurance overall pass rate: {ins_grades_df['all_pass'].mean():.0%}")
ins_grades_df

In [ ]:
# --- Engineering harness ---

def expected_condition_eng(rec):
    return "destroyed" if rec["damage_val"] == 1 else "surviving"

def expected_failure_mode_eng(rec):
    if rec["damage_val"] == 0:
        return "not applicable"
    if rec["gsi_fire"]:
        return "fire"
    if rec["gsi_tsunami"]:
        return "tsunami"
    if rec["gsi_slope_failure"]:
        return "slope"
    return "seismic"

def expected_next_step_eng(rec):
    return "no further action" if rec["damage_val"] == 0 else None  # damaged: any valid step

eng_grades = []
for rec, result in zip(test_records, engineering_results):
    txt = result["output_text"]
    condition_val = extract(txt, "Observed condition").lower()
    failure_val   = extract(txt, "Likely failure mode").lower()
    next_val      = extract(txt, "Recommended next-step").lower()
    exp_next      = expected_next_step_eng(rec)

    row = {
        "s_fid":          rec["s_fid"][-6:],
        "condition_pass": expected_condition_eng(rec) in condition_val,
        "failure_pass":   expected_failure_mode_eng(rec) in failure_val,
        "evidence_pass":  rec["conf"] in extract(txt, "Evidence basis").lower(),
        "next_step_pass": (exp_next in next_val) if exp_next else (next_val != "—"),
    }
    row["all_pass"] = all(v for k, v in row.items() if k.endswith("_pass"))
    eng_grades.append(row)

eng_grades_df = pd.DataFrame(eng_grades)
print(f"Engineering overall pass rate: {eng_grades_df['all_pass'].mean():.0%}")
eng_grades_df

In [ ]:
# --- Legal harness ---
# temperature=0.0 → deterministic output; any change across runs flags model-version drift.

def expected_condition_legal(rec):
    return "destroyed" if rec["damage_val"] == 1 else "surviving"

legal_grades = []
for rec, result in zip(test_records, legal_results):
    txt = result["output_text"]
    n_lim = len(re.findall(r"^\s*\d+[\.\)]", txt, re.MULTILINE))
    survived = rec["damage_val"] == 0

    row = {
        "s_fid":             rec["s_fid"][-6:],
        "condition_pass":    expected_condition_legal(rec) in extract(txt, "Documented condition").lower(),
        "attribution_pass":  rec["conf"] in extract(txt, "Source of attribution").lower(),
        "stmt_class_pass":   extract(txt, "Statement classes used") != "—",
        "limitations_pass":  n_lim >= 2,
        # survived buildings must carry the no-peril-attribution note
        "peril_note_pass":   ("no peril attribution" in txt.lower()) if survived else True,
    }
    row["all_pass"] = all(v for k, v in row.items() if k.endswith("_pass"))
    legal_grades.append(row)

legal_grades_df = pd.DataFrame(legal_grades)
print(f"Legal overall pass rate: {legal_grades_df['all_pass'].mean():.0%}")
legal_grades_df